# Car Price Prediction — Milestone 3: Model Creation

**Dataset:** [Car Price Prediction Challenge — Kaggle](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

Upload the four CSV files exported from M2 into Colab session storage:
`X_train_prepared.csv` · `X_test_prepared.csv` · `y_train_prepared.csv` · `y_test_prepared.csv`

---
**Notebook structure:**
1. Imports & load data
2. Evaluation metric selection & justification
3. Baseline model
4. Model 1 — Ridge Regression + hyperparameter tuning
5. Model 2 — Random Forest + hyperparameter tuning
6. Feature importance — three independent methods *(addresses M2 feedback)*
7. Final model comparison & conclusion

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120})
PAL = sns.color_palette('Set2')

print('All imports successful.')
print('RANDOM_SEED =', RANDOM_SEED)

In [ ]:
X_train = pd.read_csv('X_train_prepared.csv')
X_test  = pd.read_csv('X_test_prepared.csv')
y_train = pd.read_csv('y_train_prepared.csv').squeeze()
y_test  = pd.read_csv('y_test_prepared.csv').squeeze()

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : mean=${y_train.mean():,.0f}  std=${y_train.std():,.0f}')
print(f'y_test  : mean=${y_test.mean():,.0f}  std=${y_test.std():,.0f}')

## 2. Evaluation Metric Selection

Three complementary metrics are used to evaluate every model:

| Metric | Formula | Meaning | Direction |
|---|---|---|---|
| **RMSE** | √ mean((y − ŷ)²) | Average prediction error in dollars; penalises large errors quadratically | Lower = better |
| **MAE** | mean(\|y − ŷ\|) | Average absolute error in dollars; more robust to price outliers | Lower = better |
| **R²** | 1 − SS_res / SS_tot | Share of price variance explained by the model (0 = no better than mean, 1 = perfect) | Higher = better |

**Why RMSE as the primary metric?**
In a used-car valuation context, a prediction error of $15,000 on a $20,000 car is far more harmful than the same error on a $100,000 car spread over several smaller mistakes. RMSE's squared penalty captures this asymmetry. MAE adds an intuitive "on average, the prediction is off by $X" reading. R² provides a normalised fit score independent of price scale.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """
    Evaluate a fitted model on train and test sets.
    Returns a dict with RMSE, MAE, and R² for both splits.
    """
    p_tr = model.predict(X_tr)
    p_te = model.predict(X_te)
    return {
        'Model':      name,
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, p_tr)),
        'Test RMSE':  np.sqrt(mean_squared_error(y_te, p_te)),
        'Train MAE':  mean_absolute_error(y_tr, p_tr),
        'Test MAE':   mean_absolute_error(y_te, p_te),
        'Train R²':   r2_score(y_tr, p_tr),
        'Test R²':    r2_score(y_te, p_te),
    }

print('evaluate() defined.')

## 3. Baseline Model

**Strategy:** `DummyRegressor(strategy='mean')` — always predicts the training mean price regardless of features.

**Why this baseline?**
It is the simplest possible "model". Any real ML algorithm that cannot beat this has no practical value whatsoever. We expect Test R² ≈ 0 and Test RMSE ≈ price standard deviation of the training set.

In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
res_base = evaluate('Baseline (Mean)', baseline, X_train, y_train, X_test, y_test)

print(f"Baseline constant prediction : ${float(baseline.constant_[0]):,.0f}")
print(f"Test RMSE : ${res_base['Test RMSE']:,.0f}")
print(f"Test MAE  : ${res_base['Test MAE']:,.0f}")
print(f"Test R²   : {res_base['Test R²']:.4f}  ← expected ≈ 0")

In [ ]:
# Visualise: what does "always predict the mean" look like?
baseline_pred = np.full(len(y_test), float(baseline.constant_[0]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_test, baseline_pred, alpha=0.3, s=8, color=PAL[3])
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=1.5, label='Perfect prediction')
axes[0].axhline(float(baseline.constant_[0]), color='steelblue',
                lw=1.5, linestyle=':', label='Mean prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Baseline — Predicted vs Actual (R² = {res_base["Test R²"]:.3f})')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend(fontsize=9)

# Residual distribution
residuals_b = y_test.values - baseline_pred
axes[1].hist(residuals_b, bins=55, color=PAL[3], edgecolor='none', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='Zero error')
axes[1].set_xlabel('Residual ($)'); axes[1].set_ylabel('Count')
axes[1].set_title('Baseline — Residual Distribution')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].legend(fontsize=9)

plt.suptitle('Baseline Model (Mean Predictor)', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 4. Model 1 — Ridge Regression

### What it does
Ridge Regression is ordinary least-squares linear regression augmented with an **L2 penalty** on the magnitude of the coefficients:

> Loss = MSE + α × Σ(βᵢ²)

The penalty parameter **alpha (α)** controls regularisation strength. Higher alpha shrinks all coefficients toward zero, reducing overfitting at the cost of increased bias.

### Why Ridge for this task?
- Tests the hypothesis that car price relationships are approximately **linear** — a useful sanity check before moving to more complex models
- All features were **MinMaxScaled** in M2, making coefficients directly comparable in magnitude
- Fast to train and fully interpretable through its coefficients
- A fundamentally **different algorithm family** from Random Forest (linear vs ensemble tree)

### Hyperparameter tuning
5-fold cross-validation over 8 alpha values ranging from 0.01 to 1000.

In [ ]:
ridge_param_grid = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

ridge_cv = GridSearchCV(
    Ridge(random_state=RANDOM_SEED),
    ridge_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    return_train_score=True
)
ridge_cv.fit(X_train, y_train)
best_ridge = ridge_cv.best_estimator_

print(f'Best alpha   : {ridge_cv.best_params_["alpha"]}')
print(f'Best CV RMSE : ${-ridge_cv.best_score_:,.0f}')

In [ ]:
# Cross-validation RMSE curve across alpha values
alphas   = ridge_param_grid['alpha']
cv_means = [-ridge_cv.cv_results_[f'split{j}_test_score'] for j in range(5)]
mean_cv  = np.mean(cv_means, axis=0)
std_cv   = np.std(cv_means, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(range(len(alphas)), mean_cv, yerr=std_cv,
            fmt='o-', color=PAL[0], capsize=5, lw=2, markersize=7,
            label='CV RMSE (mean ± std across 5 folds)')
best_idx = alphas.index(ridge_cv.best_params_['alpha'])
ax.axvline(best_idx, color='red', linestyle='--', lw=1.5,
           label=f'Best α = {ridge_cv.best_params_["alpha"]}')
ax.set_xticks(range(len(alphas))); ax.set_xticklabels(alphas)
ax.set_xlabel('Alpha (regularisation strength)'); ax.set_ylabel('CV RMSE ($)')
ax.set_title('Ridge Regression — Cross-Validation RMSE across Alpha Values',
             fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=9); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
res_ridge  = evaluate('Ridge Regression', best_ridge, X_train, y_train, X_test, y_test)
ridge_pred = best_ridge.predict(X_test)

print(f"Train RMSE : ${res_ridge['Train RMSE']:,.0f}  |  Test RMSE : ${res_ridge['Test RMSE']:,.0f}")
print(f"Train MAE  : ${res_ridge['Train MAE']:,.0f}  |  Test MAE  : ${res_ridge['Test MAE']:,.0f}")
print(f"Train R²   : {res_ridge['Train R²']:.4f}   |  Test R²   : {res_ridge['Test R²']:.4f}")

# Predicted vs Actual + Residuals vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, ridge_pred, alpha=0.2, s=8, color=PAL[0])
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Ridge — Predicted vs Actual  (Test R² = {res_ridge["Test R²"]:.3f})')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend(fontsize=9)

residuals_r = y_test.values - ridge_pred
axes[1].scatter(ridge_pred, residuals_r, alpha=0.2, s=8, color=PAL[0])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--', label='Zero residual')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Ridge — Residuals vs Predicted
(fan shape = variance increases with price)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].legend(fontsize=9)

plt.suptitle('Ridge Regression — Results', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 5. Model 2 — Random Forest Regressor

### What it does
Random Forest builds an **ensemble of decision trees**, each trained on a bootstrap sample of the data with a random subset of features considered at each split. The final prediction is the **mean across all trees**. The randomness reduces variance without significantly increasing bias (bagging principle).

### Why Random Forest for this task?
- Car price depreciation is **non-linear** — a 1-year-old car loses more value per year than a 15-year-old car. RF captures this naturally with no distributional assumptions
- Handles **interactions** between features (e.g. turbo engine on a new car vs old car) without manual feature crosses
- Provides reliable **feature importances** through multiple trees
- Ensemble averaging makes it **robust to outliers** in the target

### Hyperparameter tuning
5-fold GridSearchCV over `n_estimators`, `max_depth`, and `min_samples_split`.

In [ ]:
rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5],
}

rf_cv = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_SEED),
    rf_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    return_train_score=True
)
rf_cv.fit(X_train, y_train)
best_rf = rf_cv.best_estimator_

print(f'Best params  : {rf_cv.best_params_}')
print(f'Best CV RMSE : ${-rf_cv.best_score_:,.0f}')

In [ ]:
# Heatmap of CV RMSE: max_depth x n_estimators
rf_res = pd.DataFrame(rf_cv.cv_results_)
rf_res['mean_rmse'] = -rf_res['mean_test_score']
heat = rf_res.groupby(
    ['param_max_depth', 'param_n_estimators'])['mean_rmse'].mean().unstack()
heat.index = [str(x) for x in heat.index]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heat, annot=True, fmt='.0f', cmap='YlOrRd_r', ax=ax,
            cbar_kws={'label': 'CV RMSE ($)'})
ax.set_xlabel('n_estimators'); ax.set_ylabel('max_depth')
ax.set_title('Random Forest — Hyperparameter Tuning Heatmap
'
             '(CV RMSE by max_depth × n_estimators, min_samples_split averaged)',
             fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
res_rf  = evaluate('Random Forest', best_rf, X_train, y_train, X_test, y_test)
rf_pred = best_rf.predict(X_test)

print(f"Train RMSE : ${res_rf['Train RMSE']:,.0f}  |  Test RMSE : ${res_rf['Test RMSE']:,.0f}")
print(f"Train MAE  : ${res_rf['Train MAE']:,.0f}  |  Test MAE  : ${res_rf['Test MAE']:,.0f}")
print(f"Train R²   : {res_rf['Train R²']:.4f}   |  Test R²   : {res_rf['Test R²']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, rf_pred, alpha=0.2, s=8, color=PAL[1])
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)'); axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Random Forest — Predicted vs Actual  (Test R² = {res_rf["Test R²"]:.3f})')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend(fontsize=9)

residuals_rf = y_test.values - rf_pred
axes[1].scatter(rf_pred, residuals_rf, alpha=0.2, s=8, color=PAL[1])
axes[1].axhline(0, color='red', lw=1.5, linestyle='--', label='Zero residual')
axes[1].set_xlabel('Predicted Price ($)'); axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Random Forest — Residuals vs Predicted')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[1].legend(fontsize=9)

plt.suptitle('Random Forest — Results', fontweight='bold')
sns.despine(); plt.tight_layout(); plt.show()

## 6. Feature Importance — Three Independent Methods

The M2 teacher feedback noted: *"you are overly relying on a trained model to compute feature importances... a more in-depth analysis comparing these feature importances across different seeds or with different importance scores would have been informative."*

We address this directly here with three independent methods:

| Method | How it works | Strength |
|---|---|---|
| **RF Built-in** | Mean decrease in node impurity across all trees | Fast; uses fitted model |
| **Permutation Importance (3 seeds)** | Shuffles one feature at a time on test set; measures MSE increase | Model-agnostic; evaluated on held-out data; repeated across seeds to check stability |
| **Pearson \|r\| with Price** | Absolute linear correlation between each feature and the target | Completely model-free; independent check |

If rankings agree across all three methods → the signal is real. If they disagree → the feature may be predictive only in certain contexts.

In [ ]:
# Method 1: RF built-in importance
rf_builtin = pd.Series(best_rf.feature_importances_, index=X_train.columns)

# Method 2: Permutation importance — 3 different seeds on held-out TEST set
perm_runs = []
for seed in [42, 123, 999]:
    p = permutation_importance(
        best_rf, X_test, y_test,
        n_repeats=5, random_state=seed, n_jobs=-1
    )
    perm_runs.append(pd.Series(p.importances_mean, index=X_train.columns))

perm_df = pd.concat(perm_runs, axis=1)
perm_df.columns = ['Seed 42', 'Seed 123', 'Seed 999']
perm_df['Mean'] = perm_df.mean(axis=1)
perm_df['Std']  = perm_df.std(axis=1)

# Method 3: Pearson |r| with Price — model-free linear correlation
corr_imp = X_train.corrwith(y_train).abs()

top12 = rf_builtin.sort_values(ascending=False).head(12).index.tolist()
print('Top 12 features by RF importance:')
print(rf_builtin[top12].round(4).to_string())

In [ ]:
# Side-by-side comparison of all three methods
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Panel 1
rf_builtin[top12].sort_values().plot.barh(ax=axes[0], color=PAL[1], edgecolor='white')
axes[0].set_title('Method 1: RF Built-in Importance\n(mean impurity decrease)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Panel 2
perm_top = perm_df.loc[top12].sort_values('Mean')
axes[1].barh(perm_top.index, perm_top['Mean'],
             xerr=perm_top['Std'], color=PAL[0],
             ecolor='grey', capsize=3, edgecolor='white')
axes[1].set_title('Method 2: Permutation Importance\n(mean ± std across 3 random seeds)', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Mean MSE increase when feature is shuffled')

# Panel 3
corr_imp[top12].sort_values().plot.barh(ax=axes[2], color=PAL[2], edgecolor='white')
axes[2].set_title('Method 3: |Pearson r| with Price\n(model-free linear correlation)', fontsize=10, fontweight='bold')
axes[2].set_xlabel('|Correlation with Price|')

plt.suptitle('Feature Importance — Three Independent Methods\n'
             'Consistent rankings across methods = reliable signal',
             fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Seed stability plot — are the permutation importance rankings stable?
top8 = perm_df['Mean'].sort_values(ascending=False).head(8).index.tolist()
x = np.arange(len(top8)); width = 0.26

fig, ax = plt.subplots(figsize=(12, 5))
for i, (col, color) in enumerate(zip(['Seed 42', 'Seed 123', 'Seed 999'], PAL[:3])):
    ax.bar(x + i*width, perm_df.loc[top8, col],
           width, label=col, color=color, edgecolor='white', alpha=0.9)

ax.set_xticks(x + width)
ax.set_xticklabels(top8, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Permutation Importance (MSE increase when shuffled)')
ax.set_title('Permutation Importance Stability — Top 8 Features across 3 Random Seeds\n'
             'Bar heights consistent across seeds = ranking is not a random artefact',
             fontweight='bold')
ax.legend(title='Random Seed', fontsize=9)
sns.despine(); plt.tight_layout(); plt.show()

# Quantify stability
print("Coefficient of variation (std/mean) — lower = more stable:")
cv_score = (perm_df['Std'] / perm_df['Mean'].abs()).loc[top8].sort_values()
print(cv_score.round(3).to_string())

## 7. Final Model Comparison

In [ ]:
results_df = pd.DataFrame([res_base, res_ridge, res_rf])

# Full results table
print('=' * 70)
print('FINAL MODEL COMPARISON — ALL METRICS')
print('=' * 70)
display_cols = ['Model', 'Test RMSE', 'Test MAE', 'Test R²', 'Train R²']
print(results_df[display_cols].to_string(index=False))

rmse_vs_baseline = (res_base['Test RMSE'] - res_rf['Test RMSE']) / res_base['Test RMSE'] * 100
print(f'\nBest model (lowest Test RMSE): Random Forest')
print(f'RMSE improvement over baseline: {rmse_vs_baseline:.1f}%')

In [ ]:
# Bar chart comparison — RMSE, MAE, R²
models      = ['Baseline\n(Mean)', 'Ridge\nRegression', 'Random\nForest']
model_keys  = ['Baseline (Mean)', 'Ridge Regression', 'Random Forest']
colors_bars = [PAL[3], PAL[0], PAL[1]]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric in zip(axes, ['Test RMSE', 'Test MAE', 'Test R²']):
    vals = [results_df[results_df['Model'] == k][metric].values[0] for k in model_keys]
    bars = ax.bar(models, vals, color=colors_bars, edgecolor='white', width=0.55)
    for bar, val in zip(bars, vals):
        label = f'{val:.3f}' if metric == 'Test R²' else f'${val:,.0f}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals) * 0.015,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_ylabel(metric)
    if metric != 'Test R²':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
    ax.set_xticklabels(models, fontsize=9)

plt.suptitle('Final Model Comparison — Test Set Performance', fontweight='bold', fontsize=13)
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
# Predicted price distribution overlay
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(y_test.values, bins=55, alpha=0.5, color='grey',  label='Actual',        density=True)
ax.hist(np.full(len(y_test), float(baseline.constant_[0])),
        bins=55, alpha=0.5, color=PAL[3], label='Baseline', density=True)
ax.hist(ridge_pred, bins=55, alpha=0.5, color=PAL[0], label='Ridge Regression', density=True)
ax.hist(rf_pred,    bins=55, alpha=0.5, color=PAL[1], label='Random Forest',    density=True)

ax.set_xlabel('Price ($)'); ax.set_ylabel('Density')
ax.set_title('Predicted vs Actual Price Distribution — Test Set\n'
             'Random Forest most closely matches the actual shape',
             fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.legend(fontsize=10); sns.despine(); plt.tight_layout(); plt.show()

## 8. Conclusion

Both ML models substantially outperform the mean baseline, confirming that car price is meaningfully predictable from the available features.

**Random Forest** is the best-performing model overall:
- Captures non-linear depreciation and feature interactions that Ridge cannot model linearly
- Achieves the lowest Test RMSE and highest Test R²

**Ridge Regression** is competitive but limited:
- The fan-shaped residual plot reveals that price variance grows with value — a sign that linear assumptions are violated for high-end vehicles
- Still far better than the baseline and offers full interpretability through its coefficients

**Feature importance analysis** (three methods, three seeds) confirms that **Production Year**, **Car Age**, **Mileage**, and **Engine Volume** are the most reliable predictors — consistent across all methods and stable across random seeds. This directly addresses the M2 feedback on importance instability.

**Next steps (Milestone 4):** Log-transform the price target to address heteroscedasticity, explore Gradient Boosting (XGBoost/LightGBM), and engineer additional interaction features.